<a href="https://colab.research.google.com/github/daniivelascoo/ifp-programacion-ia/blob/main/Lab_2_2_Titanic_Algoritmos_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚢 Laboratorio 2.2: Titanic - Time Travel Insurance
**Núcleo Formativo 2 - Algoritmos de Machine Learning**

---
### 📜 Contexto del Negocio
Estamos en el año 2150. Los viajes en el tiempo son turísticos. Tu empresa, **ChronoLife Insurance**, vende pólizas de vida para gente que quiere viajar al pasado y vivir el hundimiento del Titanic en primera fila.

Necesitamos una IA que calcule el **riesgo de muerte** de un cliente basándose en su perfil (clase, edad, género) para ajustar el precio de la póliza.

### 🎯 Tu Misión
1.  **Limpiar** los datos históricos (rellenar edades faltantes).
2.  **Preparar** los datos para la IA (convertir texto a números).
3.  **Entrenar** dos modelos rivales: **Regresión Logística** (El Clásico) vs **Random Forest** (El Moderno).
4.  **Decidir** cuál es mejor para el negocio.

---

In [12]:
# --- ⚙️ 0. CARGA DE DATOS (NO TOCAR) ---
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Usamos el dataset integrado en la librería Seaborn (100% fiable)
df = sns.load_dataset('titanic')

# Seleccionamos las columnas útiles para el seguro
# survived: 0=No, 1=Sí
# pclass: 1, 2, 3 (Clase del billete)
# sex: male, female
# age: Edad
# fare: Precio del billete
# alone: Booleano (Viaja solo)
cols_utiles = ['survived', 'pclass', 'sex', 'age', 'fare', 'alone']
df = df[cols_utiles]

print("✅ Manifiesto de pasajeros cargado.")
display(df.head())
print("\n--- Estado de Nulos ---")
print(df.isnull().sum())

✅ Manifiesto de pasajeros cargado.


,survived,pclass,sex,age,fare,alone
0,0,3,male,22.0,7.2500,False
1,1,1,female,38.0,71.2833,False
2,1,3,female,26.0,7.9250,True
3,1,1,female,35.0,53.1000,False
4,0,3,male,35.0,8.0500,True



--- Estado de Nulos ---
survived      0
pclass        0
sex           0
age         177
fare          0
alone         0
dtype: int64


---
## 🧹 FASE 1: Preprocesamiento (Limpieza)

Las IAs no entienden de huecos vacíos (`NaN`) ni de texto (`male`/`female`).

**Misión:**
1.  Rellena los nulos de `age` con la **mediana** de edad.
2.  Convierte la columna `sex` a números: `'male' -> 0`, `'female' -> 1`.
3.  Convierte la columna `alone` a números (de True/False a 1/0).

In [13]:
# 1. Rellenar Edad (Imputación)
mediana_edad = df['age'].mean() # Calcula la mediana
df['age'] = df['age'].fillna(mediana_edad)

# 2. Encoding de Sexo (Mapeo manual)
# Usamos un diccionario: {'male': 0, 'female': 1}
df['sex'] = df['sex'].map({'male': 0, 'female': 1})

# 3. Encoding de Alone (Booleano a Entero)
df['alone'] = df['alone'].astype(int)

# Verificación
print("Datos limpios:")
display(df.head(3))

Datos limpios:


,survived,pclass,sex,age,fare,alone
0,0,3,0,22.0,7.2500,0
1,1,1,1,38.0,71.2833,0
2,1,3,1,26.0,7.9250,1


---
## ✂️ FASE 2: Definición y Split

Separamos las características (X) del objetivo (y) y dividimos los datos.

**Misión:**
1.  Define `X` (Todo menos 'survived') e `y` ('survived').
2.  Haz el Split con `test_size=0.2` y `random_state=42`.

In [14]:
from sklearn.model_selection import train_test_split

# 1. Separar X e y
X = df.drop('survived', axis=1) # Quita la columna objetivo
y = df['survived']

# 2. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,  # 20%
    random_state=42     # Semilla
)

print(f"Pasajeros para entrenar: {X_train.shape[0]}")
print(f"Pasajeros para validar:  {X_test.shape[0]}")

Pasajeros para entrenar: 712
Pasajeros para validar:  179


---
## 🤖 FASE 3: El Duelo de Algoritmos

Vamos a entrenar dos modelos para ver cuál se comporta mejor.

*   **Aspirante 1:** `LogisticRegression`. Un modelo lineal, simple y explicable.
*   **Aspirante 2:** `RandomForestClassifier`. Un conjunto de árboles de decisión, potente y no lineal.

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# --- MODELO 1: REGRESIÓN LOGÍSTICA ---
print("Entrenando Regresión Logística...")
# Aumentamos max_iter para asegurar que converja
modelo_log = LogisticRegression(max_iter=1000)
modelo_log.fit(X_train, y_train) # TODO: Entrena (.fit)

# --- MODELO 2: RANDOM FOREST ---
print("Entrenando Random Forest...")
# Usamos n_estimators=100 (100 árboles)
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_train, y_train) # TODO: Entrena (.fit)

print("¡Modelos listos para el combate!")

Entrenando Regresión Logística...
Entrenando Random Forest...
¡Modelos listos para el combate!


---
## 📝 FASE 4: Evaluación y Decisión

Vamos a calcular la precisión (`accuracy_score`) de ambos en el examen final (Test set).

In [16]:
from sklearn.metrics import accuracy_score

# Predicciones
pred_log = modelo_log.predict(X_test)
pred_rf = modelo_rf.predict(X_test)

# Notas
nota_log = accuracy_score(y_test, pred_log)
nota_rf = accuracy_score(y_test, pred_rf)

print(f"Precisión Logística:    {nota_log:.2%}")
print(f"Precisión Random Forest: {nota_rf:.2%}")

Precisión Logística:    79.89%
Precisión Random Forest: 79.89%


### 🧐 REFLEXIÓN (Obligatorio)
*Haz doble clic para editar.*

**Pregunta:** ¿Qué modelo ha ganado? ¿Por qué crees que el Random Forest suele funcionar mejor en este tipo de datos complejos que una simple línea recta (Logística)?

**Respuesta:**

---
## ⭐ PARTE EXTRA: ¿Sobrevivirías tú? (Bonus)

Crea un pasajero con tus datos (o inventados) y pídele al mejor modelo que prediga tu destino.
*   `pclass`: 1 (Rico), 2 (Medio), 3 (Pobre)
*   `sex`: 0 (Hombre), 1 (Mujer)
*   `age`: Tu edad
*   `fare`: Precio del billete (ej. 50)
*   `alone`: 1 (Solo) o 0 (Acompañado)

In [20]:
# TU CÓDIGO AQUÍ (Define el dataframe 'yo' y usa modelo.predict)

df_yo = pd.DataFrame({
    'pclass': [1],
    'sex': [0],
    'age': [23],
    'fare': [150],
    'alone': [0]
})

pred_log_yo = modelo_log.predict(df_yo)
pred_rf_yo = modelo_rf.predict(df_yo)

print("Predicción Logística:", pred_log_yo[0])
print("Predicción Random Forest:", pred_rf_yo[0])

Predicción Logística: 1
Predicción Random Forest: 0


---
## 🏁 VALIDACIÓN FINAL
Ejecuta la celda de abajo.

In [21]:
# --- 🤖 CÓDIGO DE VALIDACIÓN (NO MODIFICAR) ---
def validar_titanic():
    print("🚀 AUDITANDO CÁLCULO DE RIESGO...\n")
    puntos = 0
    errores = []

    v_X = globals().get('X')
    v_y = globals().get('y')
    v_X_test = globals().get('X_test')
    v_log = globals().get('modelo_log')
    v_rf = globals().get('modelo_rf')

    # 1. PREPROCESO
    if v_X is not None:
        if v_X.isnull().sum().sum() == 0:
            if pd.api.types.is_numeric_dtype(v_X['sex']):
                print("✅ [FASE 1] Limpieza y Encoding: CORRECTO.")
                puntos += 2.5
            else:
                errores.append("❌ La columna 'sex' sigue siendo texto.")
        else:
            errores.append("❌ Aún hay nulos (posiblemente en edad).")
    else:
        errores.append("❌ No has definido X.")

    # 2. SPLIT
    if v_X_test is not None:
        if len(v_X_test) > 170 and len(v_X_test) < 185: # Aprox 20% de 891
            print("✅ [FASE 2] Split: CORRECTO.")
            puntos += 2.5
        else:
            errores.append("❌ El tamaño del test no parece el 20%.")
    else:
        errores.append("❌ No has definido X_test.")

    # 3. MODELOS
    if v_log is not None and v_rf is not None:
        if hasattr(v_rf, 'feature_importances_'):
            print("✅ [FASE 3] Modelos Entrenados: CORRECTO.")
            puntos += 2.5
        else:
            errores.append("❌ Los modelos no están entrenados (.fit).")
    else:
        errores.append("❌ Faltan las variables 'modelo_log' o 'modelo_rf'.")

    # 4. BONUS TRACK
    historial = "\n".join(globals().get('_ih', []))
    if 'DataFrame' in historial and 'predict' in historial and 'pclass' in historial:
        print("🌟 [BONUS] Predicción personal detectada.")
        puntos += 2.5
    else:
        print("ℹ️ [BONUS] No has hecho la predicción personal.")

    # REPORTE
    print("\n" + "="*50)
    if puntos >= 7.5:
        import hashlib
        code = hashlib.md5(str(puntos).encode()).hexdigest()[:8].upper()
        print(f"🎉 ¡PÓLIZA EMITIDA! Análisis completado.")
        print(f"🔐 CÓDIGO DE VALIDACIÓN: TIME-{code}")
        print(">> Sube el notebook a la plataforma.")
    else:
        print("⚠️ REVISA LOS ERRORES:")
        for e in errores: print(f"   - {e}")
    print("="*50)

validar_titanic()

🚀 AUDITANDO CÁLCULO DE RIESGO...

✅ [FASE 1] Limpieza y Encoding: CORRECTO.
✅ [FASE 2] Split: CORRECTO.
✅ [FASE 3] Modelos Entrenados: CORRECTO.
🌟 [BONUS] Predicción personal detectada.

🎉 ¡PÓLIZA EMITIDA! Análisis completado.
🔐 CÓDIGO DE VALIDACIÓN: TIME-43A1437F
>> Sube el notebook a la plataforma.
